In [573]:
# 客户端初始化
import voyageai
import json
# 加载环境变量并创建客户端
from dotenv import load_dotenv
from anthropic import Anthropic

embedding_client = voyageai.Client()


load_dotenv()
base_url = "http://127.0.0.1:8045"
api_key = "sk-5b2137d3a6c74956a519669d86aa4e71"
client = Anthropic(api_key=api_key, base_url=base_url)
model = "gemini-3-flash"
# model = "claude-opus-4-5-thinking"

In [574]:
# 辅助函数
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    for block in message.content:
        if block.type == "text":
            if stop_sequences:
                for stop_seq in stop_sequences:
                    if stop_seq in block.text:
                        block.text = block.text.split(stop_seq)[0]
                        break   
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [575]:
# 按章节分块
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [576]:
# 嵌入向量生成
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = embedding_client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [577]:
# VectorIndex 向量索引实现
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric 只能是 'cosine' 或 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError("初始化时未提供嵌入函数。")
        if not isinstance(document, dict):
            raise TypeError("document 必须是字典。")
        if "content" not in document:
            raise ValueError("文档字典必须包含 'content' 键。")

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("文档的 'content' 必须是字符串。")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def add_documents(self, documents: List[Dict[str, Any]]):
        if not self._embedding_fn:
            raise ValueError("初始化时未提供嵌入函数。")

        if not isinstance(documents, list):
            raise TypeError("documents 必须是字典列表。")

        if not documents:
            return

        contents = []
        for i, doc in enumerate(documents):
            if not isinstance(doc, dict):
                raise TypeError(f"下标 {i} 处的 document 必须是字典。")
            if "content" not in doc:
                raise ValueError(f"下标 {i} 处的文档必须包含 'content' 键。")
            if not isinstance(doc["content"], str):
                raise TypeError(f"下标 {i} 处文档的 'content' 必须是字符串。")
            contents.append(doc["content"])

        vectors = self._embedding_fn(contents)

        for vector, document in zip(vectors, documents):
            self.add_vector(vector=vector, document=document)

    def search(self, query: Any, k: int = 1) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError("对字符串查询未提供嵌入函数。")
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError("query 必须是字符串或数字列表。")

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"查询向量维度不匹配：期望 {self._vector_dim}，实际 {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k 必须是正整数。")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("vector 必须是数字列表。")
        if not isinstance(document, dict):
            raise TypeError("document 必须是字典。")
        if "content" not in document:
            raise ValueError("文档字典必须包含 'content' 键。")

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"向量维度不一致：期望 {self._vector_dim}，实际 {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("两个向量维度必须相同")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("两个向量维度必须相同")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("两个向量维度必须相同")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "是" if self._embedding_fn else "否"
        return f"VectorIndex(条数={len(self)}, 维度={self._vector_dim}, 度量='{self._distance_metric}', 含嵌入函数={has_embed_fn})"

In [578]:
# BM25 索引实现
from collections import Counter
from typing import Callable, Optional, Any, List, Dict, Tuple


class BM25Index:
    def __init__(
        self,
        k1: float = 1.5,
        b: float = 0.75,
        tokenizer: Optional[Callable[[str], List[str]]] = None,
    ):
        self.documents: List[Dict[str, Any]] = []
        self._corpus_tokens: List[List[str]] = []
        self._doc_len: List[int] = []
        self._doc_freqs: Dict[str, int] = {}
        self._avg_doc_len: float = 0.0
        self._idf: Dict[str, float] = {}
        self._index_built: bool = False

        self.k1 = k1
        self.b = b
        self._tokenizer = tokenizer if tokenizer else self._default_tokenizer

    def _default_tokenizer(self, text: str) -> List[str]:
        text = text.lower()
        tokens = re.split(r"\W+", text)
        return [token for token in tokens if token]

    def _update_stats_add(self, doc_tokens: List[str]):
        self._doc_len.append(len(doc_tokens))

        seen_in_doc = set()
        for token in doc_tokens:
            if token not in seen_in_doc:
                self._doc_freqs[token] = self._doc_freqs.get(token, 0) + 1
                seen_in_doc.add(token)

        self._index_built = False

    def _calculate_idf(self):
        N = len(self.documents)
        self._idf = {}
        for term, freq in self._doc_freqs.items():
            idf_score = math.log(((N - freq + 0.5) / (freq + 0.5)) + 1)
            self._idf[term] = idf_score

    def _build_index(self):
        if not self.documents:
            self._avg_doc_len = 0.0
            self._idf = {}
            self._index_built = True
            return

        self._avg_doc_len = sum(self._doc_len) / len(self.documents)
        self._calculate_idf()
        self._index_built = True

    def add_document(self, document: Dict[str, Any]):
        if not isinstance(document, dict):
            raise TypeError("document 必须是字典。")
        if "content" not in document:
            raise ValueError("文档字典必须包含 'content' 键。")

        content = document.get("content", "")
        if not isinstance(content, str):
            raise TypeError("文档的 'content' 必须是字符串。")

        doc_tokens = self._tokenizer(content)

        self.documents.append(document)
        self._corpus_tokens.append(doc_tokens)
        self._update_stats_add(doc_tokens)

    def add_documents(self, documents: List[Dict[str, Any]]):
        if not isinstance(documents, list):
            raise TypeError("documents 必须是字典列表。")

        if not documents:
            return

        for i, doc in enumerate(documents):
            if not isinstance(doc, dict):
                raise TypeError(f"下标 {i} 处的 document 必须是字典。")
            if "content" not in doc:
                raise ValueError(f"下标 {i} 处的文档必须包含 'content' 键。")
            if not isinstance(doc["content"], str):
                raise TypeError(f"下标 {i} 处文档的 'content' 必须是字符串。")

            content = doc["content"]
            doc_tokens = self._tokenizer(content)

            self.documents.append(doc)
            self._corpus_tokens.append(doc_tokens)
            self._update_stats_add(doc_tokens)

        self._index_built = False

    def _compute_bm25_score(self, query_tokens: List[str], doc_index: int) -> float:
        score = 0.0
        doc_term_counts = Counter(self._corpus_tokens[doc_index])
        doc_length = self._doc_len[doc_index]

        for token in query_tokens:
            if token not in self._idf:
                continue

            idf = self._idf[token]
            term_freq = doc_term_counts.get(token, 0)

            numerator = idf * term_freq * (self.k1 + 1)
            denominator = term_freq + self.k1 * (
                1 - self.b + self.b * (doc_length / self._avg_doc_len)
            )
            score += numerator / (denominator + 1e-9)

        return score

    def search(
        self,
        query: Any,
        k: int = 1,
        score_normalization_factor: float = 0.1,
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.documents:
            return []

        if isinstance(query, str):
            query_text = query
        else:
            raise TypeError("BM25Index 的 query 必须是字符串。")

        if k <= 0:
            raise ValueError("k 必须是正整数。")

        if not self._index_built:
            self._build_index()

        if self._avg_doc_len == 0:
            return []

        query_tokens = self._tokenizer(query_text)
        if not query_tokens:
            return []

        raw_scores = []
        for i in range(len(self.documents)):
            raw_score = self._compute_bm25_score(query_tokens, i)
            if raw_score > 1e-9:
                raw_scores.append((raw_score, self.documents[i]))

        raw_scores.sort(key=lambda item: item[0], reverse=True)

        normalized_results = []
        for raw_score, doc in raw_scores[:k]:
            normalized_score = math.exp(-score_normalization_factor * raw_score)
            normalized_results.append((doc, normalized_score))

        normalized_results.sort(key=lambda item: item[1])

        return normalized_results

    def __len__(self) -> int:
        return len(self.documents)

    def __repr__(self) -> str:
        return f"BM25Index(文档数={len(self)}, k1={self.k1}, b={self.b}, 已构建={self._index_built})"

In [579]:
# Retriever 混合检索实现（含重排序）

from typing import Any, List, Dict, Tuple, Protocol, Callable, Optional
import random
import string


class SearchIndex(Protocol):
    def add_document(self, document: Dict[str, Any]) -> None: ...

    # 提供 add_documents 批量接口，避免 VoyageAI 限流
    def add_documents(self, documents: List[Dict[str, Any]]) -> None: ...

    def search(self, query: Any, k: int = 1) -> List[Tuple[Dict[str, Any], float]]: ...


class Retriever:
    def __init__(
        self,
        *indexes: SearchIndex,
        reranker_fn: Optional[
            Callable[[List[Dict[str, Any]], str, int], List[str]]
        ] = None,
    ):
        if len(indexes) == 0:
            raise ValueError("至少需要传入一个索引")
        self._indexes = list(indexes)
        self._reranker_fn = reranker_fn

    def add_document(self, document: Dict[str, Any]):
        if "id" not in document:
            document["id"] = "".join(
                random.choices(string.ascii_letters + string.digits, k=4)
            )

        for index in self._indexes:
            index.add_document(document)

    # 提供 add_documents 批量接口，避免 VoyageAI 限流
    def add_documents(self, documents: List[Dict[str, Any]]):
        for index in self._indexes:
            index.add_documents(documents)

    def search(
        self, query_text: str, k: int = 1, k_rrf: int = 60
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not isinstance(query_text, str):
            raise TypeError("query_text 必须是字符串。")
        if k <= 0:
            raise ValueError("k 必须是正整数。")
        if k_rrf < 0:
            raise ValueError("k_rrf 必须为非负数。")

        all_results = [index.search(query_text, k=k * 5) for index in self._indexes]

        doc_ranks = {}
        for idx, results in enumerate(all_results):
            for rank, (doc, _) in enumerate(results):
                doc_id = id(doc)
                if doc_id not in doc_ranks:
                    doc_ranks[doc_id] = {
                        "doc_obj": doc,
                        "ranks": [float("inf")] * len(self._indexes),
                    }
                doc_ranks[doc_id]["ranks"][idx] = rank + 1

        def calc_rrf_score(ranks: List[float]) -> float:
            return sum(1.0 / (k_rrf + r) for r in ranks if r != float("inf"))

        scored_docs: List[Tuple[Dict[str, Any], float]] = [
            (ranks["doc_obj"], calc_rrf_score(ranks["ranks"]))
            for ranks in doc_ranks.values()
        ]

        filtered_docs = [(doc, score) for doc, score in scored_docs if score > 0]
        filtered_docs.sort(key=lambda x: x[1], reverse=True)

        result = filtered_docs[:k]

        if self._reranker_fn is not None:
            docs_only = [doc for doc, _ in result]

            for doc in docs_only:
                if "id" not in doc:
                    doc["id"] = "".join(
                        random.choices(string.ascii_letters + string.digits, k=4)
                    )

            doc_lookup = {doc["id"]: doc for doc in docs_only}
            reranked_ids = self._reranker_fn(docs_only, query_text, k)

            new_result = []
            original_scores = {id(doc): score for doc, score in result}

            for doc_id in reranked_ids:
                if doc_id in doc_lookup:
                    doc = doc_lookup[doc_id]
                    score = original_scores.get(id(doc), 0.0)
                    new_result.append((doc, score))

            result = new_result

        return result

In [580]:
# 按章节对原文分块
with open("./RAG/report_cn.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)

In [581]:
# 重排序函数：用 LLM 从候选文档中选出并排序最相关的 k 篇
def reranker_fn(docs, query_text, k):
    joined_docs = "\n".join(
        [
            f"""
        <document>
        <document_id>{doc["id"]}</document_id>
        <document_content>{doc["content"]}</document_content>
        </document>
        """
            for doc in docs
        ]
    )

    prompt = f"""
    你将收到一组文档及各自 id。请从中选出与用户问题最相关的 {k} 篇，并按相关度从高到低排序。

    用户问题：
    <question>
    {query_text}
    </question>

    候选文档：
    <documents>
    {joined_docs}
    </documents>

    请严格按以下 JSON 格式回复（仅返回 JSON，不要其他说明）：
    ```json
    {{
        "document_ids": ["id1", "id2", ...]   // 共 {k} 个文档 id，按与问题的相关度从高到低排列，最相关的排在最前
    }}
    ```
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")

    result = chat(messages, stop_sequences=["```"])

    return json.loads(text_from_message(result))["document_ids"]

In [582]:
# 创建向量索引与 BM25 索引，并用它们构建带重排序的 Retriever
vector_index = VectorIndex(embedding_fn=generate_embedding)
bm25_index = BM25Index()

retriever = Retriever(bm25_index, vector_index, reranker_fn=reranker_fn)

In [583]:
# 将所有分块加入 Retriever（内部会同步写入两个索引）；使用批量接口避免 VoyageAI 限流
retriever.add_documents([{"content": chunk} for chunk in chunks])

In [584]:
# 检索示例：查询并返回最相关的 2 个分块（经混合检索 + 重排序）
results = retriever.search("工程团队是如何处理 INC-2023-Q4-011 的？", 2)

for doc, score in results:
    print(score, "\n", doc["content"][0:200], "\n---\n")

0.03278688524590164 
 第二节：软件工程——Project Phoenix 稳定性提升

软件工程部门投入大量精力提升支撑 Project Phoenix 的核心系统的稳定性与性能。反复出现的问题，尤其是高峰负载下的 `ERR_MEM_ALLOC_FAIL_0x8007000E` 以及影响数据检索的 `TIMEOUT_QUERY_DB_0xDEADBEEF`，被列为优先处理项，对应事件成本为 INC-2023-Q4-01 
---

0.03225806451612903 
 第十节：网络安全分析——事件响应报告：INC-2023-Q4-011

网络安全运营中心成功遏制并修复了编号为 `INC-2023-Q4-011` 的定向入侵尝试。威胁情报显示，该活动与 `ShadowNet Syndicate` 威胁行为者组织的战术、技术与程序相符。初始访问通过针对财务部门人员的鱼叉式钓鱼邮件获得，可能意在获取与第三节（财务分析）相关的数据。端点检测与响应（EDR）系统在工作站 
---

